# Hudi Incremental Upsert And Time Travel Demo

Notebook này dùng để trình bày 2 điểm mạnh của `Apache Hudi` trong project:

1. `Incremental Upsert`
2. `Time Travel`

Flow demo:

- chuẩn bị bảng demo Hudi
- chạy `upsert` để update một record và insert một record mới
- đọc snapshot cũ bằng `time travel`
- đọc snapshot mới nhất
- mở các report Markdown đã sinh ra

In [1]:
from pathlib import Path
import os
import subprocess
from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "scripts").exists():
            return candidate
    raise RuntimeError("Could not find project root containing README.md and scripts/")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT={PROJECT_ROOT}")


def run_cmd(cmd: str) -> subprocess.CompletedProcess:
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result


REPORT_DIR = PROJECT_ROOT / "reports" / "hudi_demo"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"


PROJECT_ROOT=/home/dohaidang/bigdata_hudi


## 0. Kiểm tra stack và quyền ghi report

Cell này giúp kiểm tra các service chính đang chạy. Nếu trước đó từng gặp lỗi ghi report do quyền file, cell thứ hai sẽ nới quyền cho thư mục `reports/hudi_demo` để Spark container có thể cập nhật Markdown artifact.

In [2]:
run_cmd("docker compose ps spark-master spark-worker minio hive-metastore trino")

$ docker compose ps spark-master spark-worker minio hive-metastore trino
NAME             IMAGE                                      COMMAND                  SERVICE          CREATED        STATUS                 PORTS
hive-metastore   bigdata_hudi-hive-metastore                "sh -c /entrypoint.sh"   hive-metastore   31 hours ago   Up 2 hours             10000/tcp, 0.0.0.0:9083->9083/tcp, [::]:9083->9083/tcp, 10002/tcp
minio            minio/minio:RELEASE.2025-04-22T22-12-26Z   "/usr/bin/docker-ent…"   minio            2 days ago     Up 2 hours (healthy)   0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp
spark-master     spark:3.5.8-python3                        "/opt/entrypoint.sh …"   spark-master     2 hours ago    Up 2 hours             0.0.0.0:7077->7077/tcp, [::]:7077->7077/tcp, 0.0.0.0:8082->8080/tcp, [::]:8082->8080/tcp
spark-worker     spark:3.5.8-python3                        "/opt/entrypoint.sh …"   spark-worker     2 hours ago    Up 2 hours             0.

CompletedProcess(args='docker compose ps spark-master spark-worker minio hive-metastore trino', returncode=0, stdout='NAME             IMAGE                                      COMMAND                  SERVICE          CREATED        STATUS                 PORTS\nhive-metastore   bigdata_hudi-hive-metastore                "sh -c /entrypoint.sh"   hive-metastore   31 hours ago   Up 2 hours             10000/tcp, 0.0.0.0:9083->9083/tcp, [::]:9083->9083/tcp, 10002/tcp\nminio            minio/minio:RELEASE.2025-04-22T22-12-26Z   "/usr/bin/docker-ent…"   minio            2 days ago     Up 2 hours (healthy)   0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp\nspark-master     spark:3.5.8-python3                        "/opt/entrypoint.sh …"   spark-master     2 hours ago    Up 2 hours             0.0.0.0:7077->7077/tcp, [::]:7077->7077/tcp, 0.0.0.0:8082->8080/tcp, [::]:8082->8080/tcp\nspark-worker     spark:3.5.8-python3                        "/opt/entrypoint.sh …"   spark-wo

In [3]:
run_cmd("mkdir -p reports/hudi_demo && chmod 777 reports/hudi_demo && find reports/hudi_demo -maxdepth 1 -type f -name '*.md' -exec chmod 666 {} \\;")

$ mkdir -p reports/hudi_demo && chmod 777 reports/hudi_demo && find reports/hudi_demo -maxdepth 1 -type f -name '*.md' -exec chmod 666 {} \;
chmod: thay đổi quyền hạn của 'reports/hudi_demo/time_travel_summary_20260503130118455.md': Thao tác không được phép
chmod: thay đổi quyền hạn của 'reports/hudi_demo/time_travel_summary_20260503130137850.md': Thao tác không được phép



CompletedProcess(args="mkdir -p reports/hudi_demo && chmod 777 reports/hudi_demo && find reports/hudi_demo -maxdepth 1 -type f -name '*.md' -exec chmod 666 {} \\;", returncode=0, stdout='', stderr="chmod: thay đổi quyền hạn của 'reports/hudi_demo/time_travel_summary_20260503130118455.md': Thao tác không được phép\nchmod: thay đổi quyền hạn của 'reports/hudi_demo/time_travel_summary_20260503130137850.md': Thao tác không được phép\n")

## 1. Chuẩn bị bảng demo Hudi

Bước này lấy một tập nhỏ từ `payments_bronze` và ghi thành bảng demo riêng:

- bảng demo: `s3a://lakehouse/demo/payments_bronze_demo`
- mục đích: không làm bẩn các bảng chính của pipeline

In [4]:
run_cmd("bash scripts/spark_submit_container.sh pipelines/tools/prepare_hudi_payments_demo.py")

$ bash scripts/spark_submit_container.sh pipelines/tools/prepare_hudi_payments_demo.py
# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
Prepared demo table at s3a://lakehouse/demo/payments_bronze_demo
Current instant: 20260503143926792
Initial demo snapshot
+-------------------+---------------------+----------------------------------+----------------------+------------------------------------------------------------------------+--------------------------------+------------------+------------+--------------------+-------------+----------------------------------+--------------------------+---------+--------------------------------+--------------+
|_hoodie_commit_time|_hoodie_commit_seqno |_hoodie_record_key                |_hoodie_partition_path|_hoodie_file_name                                                       |order_id                        |payment_sequential|payment_type|payment_

CompletedProcess(args='bash scripts/spark_submit_container.sh pipelines/tools/prepare_hudi_payments_demo.py', returncode=0, stdout='# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf\nPrepared demo table at s3a://lakehouse/demo/payments_bronze_demo\nCurrent instant: 20260503143926792\nInitial demo snapshot\n+-------------------+---------------------+----------------------------------+----------------------+------------------------------------------------------------------------+--------------------------------+------------------+------------+--------------------+-------------+----------------------------------+--------------------------+---------+--------------------------------+--------------+\n|_hoodie_commit_time|_hoodie_commit_seqno |_hoodie_record_key                |_hoodie_partition_path|_hoodie_file_name                                                       |order_id               

## 2. Chạy incremental upsert

Bước này sẽ:

- update record `a9810da82917af2d9aefd1278f1dcfa0_1`
- insert record mới `demo_order_001_1`

Điểm cần quan sát trong output:

- `Before upsert instant`
- `After upsert instant`
- snapshot trước và sau khi ghi

In [7]:
run_cmd("bash scripts/spark_submit_container.sh pipelines/tools/run_hudi_incremental_upsert_demo.py")

$ bash scripts/spark_submit_container.sh pipelines/tools/run_hudi_incremental_upsert_demo.py
# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
Before upsert instant: 20260503144123292
Snapshot before upsert
+-------------------+---------------------+----------------------------------+----------------------+------------------------------------------------------------------------+--------------------------------+------------------+------------+--------------------+-------------+----------------------------------+--------------------------+---------------+--------------------------------+--------------+
|_hoodie_commit_time|_hoodie_commit_seqno |_hoodie_record_key                |_hoodie_partition_path|_hoodie_file_name                                                       |order_id                        |payment_sequential|payment_type|payment_installments|payment_value|payment_key        

CompletedProcess(args='bash scripts/spark_submit_container.sh pipelines/tools/run_hudi_incremental_upsert_demo.py', returncode=0, stdout='# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf\nBefore upsert instant: 20260503144123292\nSnapshot before upsert\n+-------------------+---------------------+----------------------------------+----------------------+------------------------------------------------------------------------+--------------------------------+------------------+------------+--------------------+-------------+----------------------------------+--------------------------+---------------+--------------------------------+--------------+\n|_hoodie_commit_time|_hoodie_commit_seqno |_hoodie_record_key                |_hoodie_partition_path|_hoodie_file_name                                                       |order_id                        |payment_sequential|payment_type|payme

## 3. Time travel về snapshot cũ

Cell này đọc snapshot trước khi upsert xảy ra.

In [ ]:
run_cmd("bash scripts/spark_submit_container.sh pipelines/tools/run_hudi_time_travel_demo.py --use-previous")

## 4. Time travel tới snapshot mới nhất

Cell này đọc snapshot sau khi upsert đã hoàn tất.

In [ ]:
run_cmd("bash scripts/spark_submit_container.sh pipelines/tools/run_hudi_time_travel_demo.py")

## 5. Mở các report đã sinh

Sau khi chạy xong, notebook sẽ hiển thị:

- report `incremental upsert`
- report `time travel` cũ
- report `time travel` mới

Đây là phần rất tiện để trình bày vì bạn có thể chỉ vào các giá trị khác nhau giữa hai snapshot.

In [ ]:
incremental_report = REPORT_DIR / "incremental_upsert_summary.md"
time_travel_reports = sorted(REPORT_DIR.glob("time_travel_summary_*.md"), key=lambda p: p.stat().st_mtime)

display(Markdown("## Incremental Upsert Summary"))
display(Markdown(incremental_report.read_text(encoding="utf-8")))

if len(time_travel_reports) >= 2:
    display(Markdown("## Previous Snapshot Summary"))
    display(Markdown(time_travel_reports[-2].read_text(encoding="utf-8")))

    display(Markdown("## Latest Snapshot Summary"))
    display(Markdown(time_travel_reports[-1].read_text(encoding="utf-8")))
else:
    print("Need at least two time_travel_summary_*.md files to display previous and latest snapshots.")

## 6. Điểm cần nói khi trình bày

- `Incremental upsert` cho thấy Hudi không cần overwrite cả bảng để cập nhật dữ liệu.
- `payment_key` được dùng làm `record key` để Hudi xác định record cần update.
- `_ingested_at` đóng vai trò `precombine field` để chọn bản ghi mới nhất.
- `Time travel` cho thấy cùng một bảng Hudi có thể được đọc ở hai trạng thái commit khác nhau.
- Snapshot cũ chưa có record insert mới và vẫn giữ giá trị cũ của record update.
- Snapshot mới phản ánh bản ghi đã được cập nhật và record mới đã xuất hiện.